# Swarm Seek: integrate with a larger system

This notebook is for **power users** who embed Swarm Seek inside a pipeline,
service, or experiment harness. You will see the documented `Colony` engine,
ask/tell against a simulated external evaluator (including a thread-pool
batch), phase introspection, and budget controls suitable for orchestration.

*Dual purpose:* scientific tutorial arc plus a CI pass criterion on the
orchestrated ask/tell run (`nbmake`).

## Setup

Install once with `pip install -e ".[dev]"`. Dependencies here are the Python
standard library, NumPy, and Swarm Seek (no cluster SDK). The printed version
is the one this notebook was checked against.

In [1]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

import numpy as np

from swarm_seek import ContinuousSpace, __version__
from swarm_seek.colony import Colony

print(f"swarm_seek {__version__}")

swarm_seek 0.1.0a0


## Motivation and background

Notebooks `01`–`04` use the ergonomic `ABC` façade. In a larger system you often
need:

- evaluation **outside** the optimizer process (queue, workers, lab bench)
- **metrics** (evals, phase, best-so-far) for dashboards or early stopping
- explicit **budgets** tied to wall-clock or dollar cost

The documented power-user type is `swarm_seek.colony.Colony`: same ask/tell
contract as `ABC`, with `phase` introspection. `ABC` remains the stable public
root export (`swarm_seek.__all__`); construct `Colony` when you need engine
hooks the façade does not emphasize.

Architecture reminder: space × variant × backend are orthogonal; your system
owns evaluation and should not import private (`_`-prefixed) helpers.

## Minimal example

Stand up a `Colony`, ask once, score with a tiny in-process stub, tell, and
read `phase` / `best`. Phase stays `init` until the first `tell` completes,
then advances (here to `employed`). That heartbeat is what orchestrators wrap.

In [2]:
def stub_objective(x: np.ndarray) -> np.ndarray:
    """Cheap synthetic score: Sphere on the last axis."""
    x = np.asarray(x, dtype=np.float64)
    return np.sum(x * x, axis=-1)


space = ContinuousSpace([(-2.0, 2.0)] * 3)
colony = Colony(
    space,
    variant="original",
    pop_size=8,
    limit=40,
    max_evals=200,
    seed=0,
)

print("phase before ask:", colony.phase)
x0 = colony.ask()
print("ask shape:", x0.shape, "phase while tell pending:", colony.phase)
colony.tell(stub_objective(x0))
print("after first tell: phase=", colony.phase, "best=", colony.best.fitness)

phase before ask: init
ask shape: (8, 3) phase while tell pending: init
after first tell: phase= employed best= 1.0550880009169372


## Progressive deep dive

### Simulated external evaluator

Model a service that accepts a batch of parameter rows and returns scores in
**ask order**. A thread pool stands in for I/O-bound workers; it is not a
speedup claim for tiny NumPy math.

In [3]:
@dataclass
class EvalService:
    """Stand-in for a remote/worker evaluation backend."""

    n_calls: int = 0
    n_points: int = 0

    def evaluate_batch(self, candidates: np.ndarray) -> np.ndarray:
        candidates = np.asarray(candidates, dtype=np.float64)
        if candidates.size == 0:
            return np.zeros(0, dtype=np.float64)

        def _one(row: np.ndarray) -> float:
            return float(np.sum(row * row))

        with ThreadPoolExecutor(max_workers=4) as pool:
            parts = list(pool.map(_one, candidates))
        fitness = np.asarray(parts, dtype=np.float64)
        self.n_calls += 1
        self.n_points += int(fitness.shape[0])
        return fitness


service = EvalService()
demo = np.array([[1.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=np.float64)
print("service batch:", service.evaluate_batch(demo), "calls=", service.n_calls)

service batch: [1. 0.] calls= 1


### Orchestration loop with metrics

Drive the colony to convergence and append a metrics row each time
``n_evals // 250`` increases (first row may be soon after init). Keep
`stall_evals` available so flat landscapes can stop early in real deployments.
Call `configure_budgets` only while `n_evals == 0`.

**Verification:** seeded run finishes with finite best fitness `< 1e-2`, at
least one metrics snapshot, and `service_points == colony.n_evals`.

In [4]:
SEED = 2
PASS_THRESHOLD = 1e-2
LOG_EVERY = 250

colony = Colony(
    ContinuousSpace([(-2.0, 2.0)] * 4),
    variant="gabc",
    pop_size=12,
    limit=60,
    max_evals=2_500,
    stall_evals=800,
    seed=SEED,
    C=1.5,
)
colony.configure_budgets(max_evals=2_500, stall_evals=800)

service = EvalService()
metrics: list[tuple[int, str, float]] = []
last_bucket = -1

while not colony.converged:
    candidates = colony.ask()
    if candidates.size == 0:
        colony.tell([])
    else:
        colony.tell(service.evaluate_batch(candidates))

    bucket = colony.n_evals // LOG_EVERY
    if colony.n_evals > 0 and bucket > last_bucket:
        last_bucket = bucket
        metrics.append(
            (colony.n_evals, colony.phase, float(colony.best.fitness))
        )

best = colony.best
print(
    f"done: fitness={best.fitness:.6e}, n_evals={colony.n_evals}, "
    f"n_iters={colony.n_iters}, service_calls={service.n_calls}, "
    f"service_points={service.n_points}"
)
print("metrics (n_evals, phase, best):")
for row in metrics:
    print(" ", row)

assert np.isfinite(best.fitness)
assert best.fitness < PASS_THRESHOLD
assert len(metrics) >= 1
assert service.n_points == colony.n_evals
print("PASS: orchestrated Colony ask/tell with external-style evaluator")

done: fitness=2.121993e-22, n_evals=2506, n_iters=125, service_calls=253, service_points=2506
metrics (n_evals, phase, best):
  (12, 'employed', 0.9104076247076285)
  (256, 'onlooker', 0.0011620009949991769)
  (503, 'onlooker', 2.7487847779371583e-05)
  (756, 'onlooker', 2.543094635884701e-07)
  (1001, 'scout', 2.987758111020828e-10)
  (1255, 'scout', 8.631247934634487e-13)
  (1506, 'onlooker', 8.029210309578793e-15)
  (1759, 'onlooker', 1.4667341578913025e-16)
  (2005, 'onlooker', 2.241394997287027e-19)
  (2252, 'scout', 2.5126772391101842e-21)
  (2506, 'scout', 2.1219931698004225e-22)
PASS: orchestrated Colony ask/tell with external-style evaluator


## Results

From the orchestration cell above, check:

- **`fitness`:** should be well below `1e-2` on this Sphere-like landscape
  (origin is the minimizer).
- **`service_points == n_evals`:** every scored ask row was billed through the
  stand-in service (empty asks do not increment evals).
- **`metrics`:** one row when each integer bucket ``n_evals // 250`` is first
  reached (so the opening row can appear at ``n_evals < 250``, e.g. after the
  initial population tell); later rows step by ~250 evals. ``phase`` after init
  is typically ``employed``, ``onlooker``, or ``scout``; best fitness should not
  increase across snapshots on this minimize run.
- **`configure_budgets`:** succeeds while `n_evals == 0`; it raises after the
  first scoring `tell`.

No plot is required; the metrics table is the integration evidence.

## Interpretation / discussion

Embedding Swarm Seek means treating the colony as a **stateful proposer**, not
as the process that owns simulation:

- Map `ask` → enqueue jobs; map completed results → `tell` in **ask order**
  (or rebuild the fitness vector to match the last ask).
- Prefer `Colony` when you need `phase` or tighter control; prefer `ABC` when
  callers should stay on the public root API.
- The thread pool only illustrates how an orchestrator might fan out work; for
  multi-node jobs use your stack (batch scheduler, Ray, Dask, …) **outside**
  Swarm Seek.

Limits: this notebook does not cover checkpoint/resume, distributed consensus,
or sklearn/Optuna adapters (post-`0.1.0`). Do not depend on private attributes.

## Takeaways

- **When to use `Colony`:** orchestrators that need phase/metrics and own
  evaluation scheduling.
- **When to stay on `ABC`:** simple scripts and the stable `swarm_seek.__all__`
  surface.
- **When not to force this pattern:** single-file experiments with a fast
  in-process objective—`ABC.minimize` is enough (`04_custom_objective.ipynb`).
- **Gotchas:** never `ask` twice without `tell`; empty batches need `tell([])`;
  `configure_budgets` only while `n_evals == 0`; fitness length must equal
  `n_ask`.

## Further reading

- Architecture (layers, ask/tell):
  https://swarm-seek.readthedocs.io/en/latest/architecture.html
- API (`ABC`, `Colony`):
  https://swarm-seek.readthedocs.io/en/latest/api.html
- Custom fitness tutorial: `examples/04_custom_objective.ipynb`
- Ask/tell on Sphere: `examples/01_ask_tell_sphere.ipynb`
- Artificial Bee Colony background (optional):
  Karaboga & Basturk (2007),
  https://doi.org/10.1007/s10898-007-9149-x